# Uitwerking 26.3 - Ontwerp een veilige MCP-toolgrens

De voorbeeldoplossing in `uitwerking.py` modelleert voor iedere tool een afzonderlijk beleid. De toolnaam, vereiste scope, toegestane parameters en de eis voor menselijke goedkeuring worden buiten het model vastgelegd. Daardoor kan tekst uit een document deze autorisatieregels niet aanpassen.

De drie fictieve tools zijn:

- `document_search`: alleen lezen met scope `documents.read`.
- `ticket_create`: schrijfactie met scope `tickets.write`, approval en een idempotency-key.
- `account_status_update`: gevoelige schrijfactie met scope `accounts.status.write`, approval en een idempotency-key.

`authorize_tool_call()` controleert vervolgens of de tool bestaat, of de gebruiker de juiste scope heeft, of alleen toegestane parameters worden gebruikt en of bij schrijfacties de vereiste approval en idempotency-key aanwezig zijn. Wanneer de aanvraag rechtstreeks voortkomt uit onbetrouwbare documentinhoud, wordt een schrijfactie geweigerd.

Dit sluit aan op de controles uit het boek: rechten worden technisch begrensd, onbetrouwbare content wordt als data behandeld en kan geen autorisatiebeslissing overschrijven, en gevoelige schrijfacties vereisen aanvullende controle.

## Aanvullende controle

Een productie-implementatie moet daarnaast de workflow zelf begrenzen met een maximum aantal stappen en een expliciete stopconditie. Voeg die begrenzing toe aan de orchestrator of agentloop en niet alleen aan de afzonderlijke toolpolicy.

## Zelf verder testen

Verander minimaal één scope, verwijder een idempotency-key en probeer een niet-toegestane parameter. Controleer dat iedere onveilige variant wordt geweigerd.


In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass(frozen=True)
class ToolPolicy:
    name: str
    scope: str
    write_action: bool
    allowed_parameters: set[str]
    approval_required: bool


POLICIES = {
    "document_search": ToolPolicy(
        name="document_search",
        scope="documents.read",
        write_action=False,
        allowed_parameters={"query", "max_results"},
        approval_required=False,
    ),
    "ticket_create": ToolPolicy(
        name="ticket_create",
        scope="tickets.write",
        write_action=True,
        allowed_parameters={"title", "description", "idempotency_key"},
        approval_required=True,
    ),
    "account_status_update": ToolPolicy(
        name="account_status_update",
        scope="accounts.status.write",
        write_action=True,
        allowed_parameters={"account_id", "status", "idempotency_key"},
        approval_required=True,
    ),
}


@dataclass
class RequestContext:
    user_scopes: set[str]
    approved: bool = False
    source_is_untrusted: bool = False
    log: list[str] = field(default_factory=list)


def authorize_tool_call(
    tool_name: str,
    arguments: dict[str, Any],
    context: RequestContext,
) -> tuple[bool, str]:
    policy = POLICIES.get(tool_name)
    if policy is None:
        return False, "Onbekende tool"

    if policy.scope not in context.user_scopes:
        return False, "Ontbrekende scope"

    unexpected = set(arguments) - policy.allowed_parameters
    if unexpected:
        return False, f"Niet-toegestane parameters: {sorted(unexpected)}"

    if policy.write_action and "idempotency_key" not in arguments:
        return False, "Idempotency-key ontbreekt"

    if policy.approval_required and not context.approved:
        return False, "Menselijke goedkeuring vereist"

    # Content uit documenten is data, geen autorisatie-instructie.
    if context.source_is_untrusted and policy.write_action:
        return False, "Schrijfactie mag niet rechtstreeks uit onbetrouwbare content volgen"

    context.log.append(f"ALLOW {tool_name}")
    return True, "Toegestaan"


if __name__ == "__main__":
    ctx = RequestContext(
        user_scopes={"documents.read", "tickets.write"},
        approved=False,
        source_is_untrusted=True,
    )

    ok, reason = authorize_tool_call(
        "ticket_create",
        {
            "title": "Controleer account",
            "description": "Afkomstig uit document",
            "idempotency_key": "demo-001",
        },
        ctx,
    )
    print(ok, reason)
